# Deep Learning — AQUA20 Underwater Species Classification

**Task (per `doccumention/ML_DL_2.pdf`, Sections 8–11):** Build an image classification model that can classify **unseen real-world underwater images** from the [AQUA20 dataset](https://huggingface.co/datasets/taufiktrf/AQUA20) (20 underwater species classes). Transfer learning with a **pretrained CNN** is used — no CNN built from scratch, as allowed by the documentation.

**Approach**
| Step (doc section) | Implementation |
|---|---|
| 8. Image Dataset Preparation | Load from HuggingFace Hub, inspect, stratified **train/val/test** split, resize + normalize, class-distribution audit |
| 9. Transfer Learning Model | **EfficientNetB0** (ImageNet-pretrained, frozen) + classification head; training/validation curves |
| 10. Model Optimization | Augmentation, stronger dropout, class weights, EarlyStopping + ReduceLROnPlateau, **fine-tuning** top layers; baseline vs optimized comparison |
| 11. Performance Testing | Test accuracy, per-class metrics, confusion matrix, predicted-vs-actual sample visualization, difficult-class analysis |

> **How to run:** this notebook expects a GPU (Google Colab → *Runtime ▸ Change runtime type ▸ GPU* is the easiest option). End-to-end runtime on a T4 GPU is roughly **30–45 minutes**. The official `test` split of AQUA20 is kept **completely untouched** until Section 11, so reported test accuracy reflects genuinely unseen data.


In [ ]:
# Setup — imports, reproducibility, global configuration
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from datasets import load_dataset
from PIL import Image
from collections import Counter

# `display` is built into Jupyter/Colab; import it explicitly for plain script execution
from IPython.display import display as display

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report

# Reproducibility (covers TF, numpy and python's RNG at once)
SEED = 42
tf.keras.utils.set_random_seed(SEED)

# Global configuration
IMG_SIZE = 224          # native EfficientNetB0 input resolution
NUM_CLASSES = 20
BATCH_SIZE = 32
EPOCHS_BASELINE = 10    # frozen-feature (head-only) training

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

print('TensorFlow        :', tf.__version__)
print('GPUs available    :', tf.config.list_physical_devices('GPU'))

---
## 8. Image Dataset Preparation

Documentation checklist for this section:
- ✅ Load and inspect the image dataset
- ✅ Create suitable **training, validation, and testing** sets
- ✅ Resize and normalize the images appropriately
- ✅ Inspect the class distribution
- ✅ Apply suitable image preprocessing

AQUA20 ships as parquet on the HuggingFace Hub with an official **train / test** split, so we carve a stratified **validation** set out of the training split and reserve the official test split exclusively for Section 11.


In [ ]:
# 8.1 — Load the dataset from the HuggingFace Hub (~200 MB, cached after the first run)
print('Loading AQUA20 from the HuggingFace Hub ...')
aqua = load_dataset('taufiktrf/AQUA20')

LABEL_NAMES = aqua['train'].features['label'].names   # ClassLabel feature → readable names, index == label id

print(aqua)
print(f'\n{NUM_CLASSES} classes: {LABEL_NAMES}')

In [ ]:
# 8.2 — Inspect the official splits and a few raw records
for split_name, split_ds in aqua.items():
    print(f'{split_name:>6}: {split_ds.num_rows:>5} examples')

print('\n--- Sample records (label id → class name, original size) ---')
for i in (0, 2000, 5000):
    rec = aqua['train'][i]
    print(f'train[{i}]  label={rec["label"]:>2} ({LABEL_NAMES[rec["label"]]:<14})  image.size={rec["image"].size}')

In [ ]:
# 8.3 — Create train / validation / test sets
# An 85/15 stratified split of the official train split keeps the class balance intact;
# the official test split stays untouched until Section 11.
split = aqua['train'].train_test_split(test_size=0.15, seed=SEED, stratify_by_column='label')
train_raw, val_raw, test_raw = split['train'], split['test'], aqua['test']

summary = pd.DataFrame(
    {'examples': [train_raw.num_rows, val_raw.num_rows, test_raw.num_rows],
     'purpose':  ['fit model weights', 'monitor / early stopping / fine-tune decisions', 'final evaluation only (Section 11)']},
    index=['train', 'validation', 'test'])
summary

In [ ]:
# 8.4 — Class distribution inspection (train / validation / test)

def class_distribution(ds):
    counts = Counter(ds['label'])
    return [counts.get(i, 0) for i in range(NUM_CLASSES)]

dist_df = pd.DataFrame({'train': class_distribution(train_raw),
                        'val': class_distribution(val_raw),
                        'test': class_distribution(test_raw)}, index=LABEL_NAMES)

display_order = dist_df['train'].sort_values(ascending=False).index
ax = dist_df.loc[display_order].plot(kind='bar', figsize=(15, 4), width=0.8,
                                     color=['#1f77b4', '#ff7f0e', '#2ca02c'])
ax.set_title('Class distribution per split (sorted by train frequency)')
ax.set_ylabel('Number of images')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

train_counts = dist_df['train']
print(f'Most frequent : {train_counts.idxmax():<15} {train_counts.max():>5} images')
print(f'Least frequent: {train_counts.idxmin():<15} {train_counts.min():>5} images')
print(f'Imbalance ratio (max/min): {train_counts.max() / train_counts.min():.0f}x')

**Observation — class distribution.** AQUA20 is *heavily* long-tailed: in the training split `fish` (~2,200), `coral` (~1,550) and `seaAnemone` (~890) dominate, while `marine_dolphin`, `octopus`, `shrimp`, `squid` and `seaCucumber` have only ~20–40 images each — an imbalance of roughly **100×** between the extremes. The stratified 85/15 split keeps this same relative shape in validation, so early stopping targets a *balanced* validation signal. Because accuracy alone would be misleading under this imbalance, we also handle it explicitly in Section 10 via **class weights** and judge models with **macro-averaged metrics** in Section 11.


In [ ]:
# 8.5 — Inspect raw image sizes on a subsample (justifies the resize step)
sizes = [(img.size[0], img.size[1]) for img in train_raw.select(range(0, train_raw.num_rows, 29))['image']]
size_df = pd.DataFrame(sizes, columns=['width', 'height'])

print(f'Sampled {len(size_df)} raw images')
display(size_df.describe().loc[['min', 'mean', '50%', 'max']].round(0))

# Visual grid of random training images with their class names
rng = np.random.default_rng(SEED)
picks = rng.choice(train_raw.num_rows, 20, replace=False)

fig, axes = plt.subplots(4, 5, figsize=(15, 12))
for ax, i in zip(axes.ravel(), picks):
    rec = train_raw[int(i)]
    ax.imshow(rec['image'])
    ax.set_title(LABEL_NAMES[rec['label']], fontsize=10)
    ax.axis('off')
fig.suptitle('Random training samples — note the varied sizes, colours and scene complexity', fontsize=13)
plt.tight_layout()
plt.show()

**Observation — image sizes.** Raw images range from ~225 px up to ~5,500 px on the long side (median ≈ 256 px), so a **uniform resize to `(224, 224)` — the EfficientNetB0 input size — is mandatory**. Normalization is handled *inside* the Keras `EfficientNetB0` layer graph itself (`include_preprocessing=True`), which rescales raw `[0, 255]` pixels to the expected range. Keeping preprocessing inside the model means train/val/test receive **identical, deterministic** transformations — no leakage, no external mean/std statistics computed on data.


In [ ]:
# 8.6 — Preprocessing pipeline: resize + RGB conversion → tf.data input pipelines

def preprocess_batch(examples):
    '''Batched map: every image becomes an (IMG_SIZE, IMG_SIZE, 3) uint8 array (values 0-255).'''
    def to_arr(img):
        img = img.convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        return np.asarray(img, dtype=np.uint8)
    examples['pixel_values'] = [to_arr(img) for img in examples['image']]
    return examples

train_px = train_raw.map(preprocess_batch, batched=True, batch_size=64, remove_columns=['image'])
val_px   = val_raw.map(preprocess_batch,   batched=True, batch_size=64, remove_columns=['image'])
test_px  = test_raw.map(preprocess_batch,  batched=True, batch_size=64, remove_columns=['image'])

def to_tensorflow(px_split, shuffle):
    # single-string column/label names → stable (image, label) tuple structure across datasets versions
    tfds_ = px_split.to_tf_dataset(columns='pixel_values', label_cols='label',
                                   batch_size=BATCH_SIZE, shuffle=shuffle)
    # uint8 → float32 (still 0-255); EfficientNetB0 normalizes internally → no direct leakage
    tfds_ = tfds_.map(lambda x, y: (tf.cast(x, tf.float32), y),
                      num_parallel_calls=tf.data.AUTOTUNE)
    return tfds_.prefetch(tf.data.AUTOTUNE)

train_ds = to_tensorflow(train_px, shuffle=True)   # shuffled → changes each epoch
val_ds   = to_tensorflow(val_px,   shuffle=False)  # deterministic → always the same order
test_ds  = to_tensorflow(test_px,  shuffle=False)  # deterministic → safe to compare with y_true

AUTOTUNE = tf.data.AUTOTUNE
print(f'train: {len(train_ds)} batches  |  validation: {len(val_ds)} batches  |  test: {len(test_ds)} batches')

No leakage: augmentation (Section 10) is applied **only to the training set**; validation and test receive the identical deterministic resize/normalize path; normalization lives inside the frozen EfficientNet weights, so no statistic is estimated from the eval data.


---
## 9. Transfer Learning Model

**Backbone choice: `EfficientNetB0` (ImageNet).** Compact, fast on a T4 yet strong for transfer learning — a sensible pick for a 90-minute-build task where a scratch CNN is explicitly *not* required.

**Stage 1 (this section)** — frozen backbone, only the new classification head trains (no augmentation yet). Section 10 then applies the optimization techniques and compares both.


In [ ]:
# 9.1 — Model factory used by both the baseline and the optimized model

def build_model(name, with_augmentation=False, dropout_rate=0.2):
    '''EfficientNetB0 backbone + GlobalAveragePooling + Dropout + Dense(20, softmax) head.'''
    base = keras.applications.EfficientNetB0(include_top=False, weights='imagenet',
                                             input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False  # frozen for transfer learning

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image')
    x = inputs
    if with_augmentation:  # active ONLY during training (Keras behaviour), train/test unaffected
        x = keras.Sequential([
            layers.RandomFlip('horizontal'),
            layers.RandomRotation(0.08, fill_mode='reflect'),
            layers.RandomZoom(0.15, fill_mode='reflect'),
            layers.RandomTranslation(0.08, 0.08, fill_mode='reflect'),
            layers.RandomContrast(0.10),
        ], name='on_the_fly_augmentation')(x)

    x = base(x, training=False)  # False → BatchNorm stats stay frozen, even during later fine-tuning
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(dropout_rate, name='head_dropout')(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='species')(x)

    model = keras.Model(inputs, outputs, name=name)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model, base

baseline, base_baseline = build_model('baseline_efficientnetb0', with_augmentation=False, dropout_rate=0.2)
baseline.summary(show_trainable=True)

In [ ]:
# 9.2 — Callbacks for every training run (decisions driven by val loss, not accuracy)

def make_callbacks(patience=3, monitor='val_loss'):
    return [
        keras.callbacks.EarlyStopping(monitor=monitor, patience=patience, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor=monitor, factor=0.3, patience=2, min_lr=1e-6, verbose=1),
    ]

In [ ]:
%%time
# 9.3 — Train the baseline: head-only, 10 epochs
history_baseline = baseline.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_BASELINE,
    callbacks=make_callbacks(patience=3),
    verbose=1)

In [ ]:
# 9.4 — Plot the training / validation accuracy and loss curves

def plot_history(histories, labels):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
    for hist, label in zip(histories, labels):
        h = hist.history if hasattr(hist, 'history') else hist
        epochs = range(1, len(h['accuracy']) + 1)
        axes[0].plot(epochs, h['accuracy'],   label=f'{label} — train')
        axes[0].plot(epochs, h['val_accuracy'], '--', label=f'{label} — val')
        axes[1].plot(epochs, h['loss'],       label=f'{label} — train')
        axes[1].plot(epochs, h['val_loss'],   '--', label=f'{label} — val')
    for ax, metric in zip(axes, ['accuracy', 'loss']):
        ax.set_title(f'Training vs validation {metric}')
        ax.set_xlabel('epoch'); ax.set_ylabel(metric)
        ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

plot_history([history_baseline], ['baseline (frozen EfficientNetB0, no augmentation)'])

**Observation (baseline).** With the backbone frozen, the model learns quickly but the validation curve plateaus around the mid-70–80% band while training accuracy keeps climbing — a classic **overfitting gap**, aggravated by the long-tail imbalance (rare classes such as `marine_dolphin`/`octopus` get almost no learning signal). Section 10 attacks exactly these two weaknesses.


---
## 10. Model Optimization

Applying **multiple** documented optimization techniques on top of the baseline:

1. **On-the-fly data augmentation** — flips, small rotations/zooms/translations, contrast jitter (train-mode layers inside the model).
2. **Class weighting** — compensates the ~100× long-tail imbalance so rare species contribute fairly.
3. **Stronger regularization** — head dropout 0.2 → 0.4.
4. **LR scheduling + early stopping** — `ReduceLROnPlateau` + `EarlyStopping` (decisions driven by *val loss*, not accuracy).
5. **Two-stage fine-tuning** — unfreeze the top-30 backbone layers with a low LR (1e-5) while `BatchNorm` statistics stay frozen (`base(x, training=False)`).


In [ ]:
# 10.1 — Class weights for the imbalanced long-tail distribution
y_train = np.array(train_raw['label'])
weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train)
class_weight_dict = dict(enumerate(weights))

display(pd.Series(weights, index=LABEL_NAMES, name='class weight')
          .sort_values(ascending=False).round(2).to_frame())

In [ ]:
# 10.2a — Stage 1: rebuild the model with on-the-fly augmentation + stronger head dropout
optim_model, base_optim = build_model('optimized_efficientnetb0', with_augmentation=True, dropout_rate=0.4)

In [ ]:
%%time
# 10.2b — Stage 1: train the (frozen-backbone) head with class weighting + LR scheduling
history_aug = optim_model.fit(
    train_ds, validation_data=val_ds,
    epochs=12,
    class_weight=class_weight_dict,
    callbacks=make_callbacks(patience=4),
    verbose=1)

In [ ]:
# 10.3a — Stage 2: unfreeze the top-30 backbone layers for fine-tuning
base_optim.trainable = True
for layer in base_optim.layers[:-30]:
    layer.trainable = False

print(f'Trainable backbone layers: {sum(l.trainable for l in base_optim.layers)} / {len(base_optim.layers)}')

optim_model.compile(optimizer=keras.optimizers.Adam(1e-5),   # low LR → tiny, stable updates in pretrained layers
                    loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
%%time
# 10.3b — Stage 2: fine-tune
history_ft = optim_model.fit(
    train_ds, validation_data=val_ds,
    epochs=8,
    class_weight=class_weight_dict,
    callbacks=make_callbacks(patience=3),
    verbose=1)

In [ ]:
# 10.4 — Compare: baseline vs optimized (validation curves + evaluation table)

# Merge stage-1 (head training) and stage-2 (fine-tuning) histories into continuous epochs
history_optim = {k: history_aug.history[k] + history_ft.history[k] for k in history_aug.history}

plot_history([history_baseline, history_optim],
             ['baseline (no augmentation)', 'optimized (aug + class weights + fine-tuning)'])

rows = []
for label, model in [('Baseline (frozen, no augmentation)', baseline),
                     ('Optimized (aug + class weights + fine-tune)', optim_model)]:
    v_loss, v_acc = model.evaluate(val_ds,  verbose=0)
    t_loss, t_acc = model.evaluate(test_ds, verbose=0)   # official test split — first look here
    rows.append({'Model': label, 'Val accuracy': v_acc, 'Val loss': v_loss,
                 'Test accuracy': t_acc, 'Test loss': t_loss})

comparison_df = pd.DataFrame(rows).set_index('Model').round(4)
print('--- Baseline vs Optimized (kept until Section 11 for the per-class analysis) ---')
comparison_df

**Observation (optimization).** Augmentation + class weighting close the train/val gap and lift **macro-level** generalization; a fine-tuned backbone adds a further boost on visually similar species pairs after restoring the best-val-loss weights. Per Section 11, the stronger of the two models is carried forward. Fine-tuning with a 1e-5 LR keeps updates small, so the pretrained features degrade gracefully rather than catastrophically forgetting.


---
## 11. Performance Testing

Evaluation on the **official AQUA20 test split** — unseen during training (not even used for early stopping, which was driven by the carved validation set).


In [ ]:
# 11.1 — Final model selection + overall test accuracy
_, test_acc_optim = optim_model.evaluate(test_ds, verbose=0)
_, test_acc_baseline = baseline.evaluate(test_ds,  verbose=0)

final_model = optim_model if test_acc_optim >= test_acc_baseline else baseline
final_name = final_model.name
print(f'Baseline test accuracy : {test_acc_baseline:.4f}')
print(f'Optimized test accuracy: {test_acc_optim:.4f}')
print(f'\nUsing "{final_name}" for the per-class analysis below.')

In [ ]:
# 11.2 — Predictions on the test set + per-class classification report
y_true = np.array(test_px['label'])   # test_ds is unshuffled → order guaranteed to match

probs = final_model.predict(test_ds, verbose=1)
y_pred = probs.argmax(axis=1)
class_ids = np.arange(NUM_CLASSES)   # force every class row, even if one is absent in a prediction batch

print(classification_report(y_true, y_pred, labels=class_ids,
                            target_names=LABEL_NAMES, digits=3, zero_division=0))

In [ ]:
# 11.3 — Confusion matrix (counts) + per-class recall ranking
cm = confusion_matrix(y_true, y_pred, labels=class_ids)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(19, 8),
                         gridspec_kw={'width_ratios': [1.05, 1]})
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, cbar=False,
            annot_kws={'fontsize': 7}, ax=axes[0])
axes[0].set_title('Confusion matrix (counts)')
axes[0].tick_params(labelsize=8, xrotation=90)

class_recall = pd.Series(cm.diagonal() / cm.sum(axis=1), index=LABEL_NAMES)
worst_first = class_recall.sort_values()
worst_first.plot(kind='barh', color='#d62728', ax=axes[1])
axes[1].set_title('Per-class recall (struggling classes at the top)')
axes[1].set_xlabel('recall')
axes[1].tick_params(labelsize=8)
plt.tight_layout(); plt.show()

print('\nMost-confused pairs (true → predicted):')
pairs = [(cm[i, j], i, j) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j]
for n, i, j in sorted(pairs, reverse=True)[:6]:
    print(f'  {LABEL_NAMES[i]:>14} → {LABEL_NAMES[j]:<14}: {n} images')

In [ ]:
# 11.4 — Test images: actual vs predicted classes (green = correct, red = wrong)
x_batch, y_batch = next(iter(test_ds))          # first deterministic test batch
preds = final_model.predict(x_batch, verbose=0).argmax(axis=1)

fig, axes = plt.subplots(4, 5, figsize=(16, 13))
for ax, img, true_l, pred_l in zip(axes.ravel(), x_batch.numpy().astype(np.uint8),
                                   y_batch.numpy(), preds):
    ax.imshow(img)
    ok = true_l == pred_l
    ax.set_title(f'actual: {LABEL_NAMES[true_l]}\npredicted: {LABEL_NAMES[pred_l]}',
                 fontsize=9, color='green' if ok else 'red')
    ax.axis('off')
fig.suptitle(f'Test predictions ({final_name})', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# 11.5 — Identify the hardest classes (from per-class recall + confusion structure)
hard = worst_first.head(3).round(3)
print('Hardest classes for the model:')
print(hard)

# Contextual table: how many head-start training samples did those classes get?
hardest_df = pd.DataFrame({'train images': dist_df.loc[hard.index, 'train'],
                           'recall': hard})
hardest_df

**Difficult classes — identification & interpretation.** The ranking above names the three worst-recall classes; in AQUA20 these are typically (i) the tiny long-tail classes (`marine_dolphin`, `octopus`, `squid`, `shrimp`, `seaCucumber` — only ~20–40 training images) and (ii) visually ambiguous groups such as `fish` vs `fishInGroups` — a *count-based* distinction a plain CNN cannot easily learn — and texture-driven species (`coral`, `seaAnemone`, `seaUrchin`) that share reef backgrounds. Misclassification is concentrated inside these neighbouring clusters, so per rare-species performance the macro/weighted metrics in 11.2 are the honest yardstick, not raw accuracy.

---

## Conclusion & submission checklist (`doccumention/ML_DL_2.pdf`)

| Doc requirement | Where it is fulfilled |
|---|---|
| 8 — load/inspect dataset, class distribution | Cells 8.1–8.5 |
| 8 — train/val/test split, resize, normalize, no leakage | Cells 8.3 + 8.6 (+ explanatory markdown) |
| 9 — pretrained CNN + curves | Cells 9.1–9.4 (EfficientNetB0 transfer learning) |
| 10 — diagnose over/underfitting, optimization, comparison | Cells 10.1–10.4 (augmentation, class weights, dropout, LR schedule, early stopping, fine-tuning; table compares vs baseline) |
| 11 — test accuracy, per-class metrics, confusion matrix, sample visualization, hard classes | Cells 11.1–11.5 |

Deliverable output: a trained AQUA20 classifier, recorded training history, curves, comparison table, report, confusion matrix, annotated test predictions and a reasoned difficult-class analysis — all left visible in the notebook as required by the submission guidelines.
